In [0]:
%sql
use catalog demo_workspace;
use schema book_store

In [0]:
%run /Workspace/Users/erica.roque@gmail.com/Databricks-Certified-Data-Engineer-Associate/Includes/Copy-Datasets

In [0]:
%python
df = spark.readStream.table("books")

df.createOrReplaceTempView("books_streaming_tmp_vw")

# Usa o caminho do teu Volume no Unity Catalog
display(
    df,
    checkpointLocation="/Volumes/demo_workspace/book_store/books/checkpoints/books_3"
)

In [0]:
%python
# 1. Lê a visão temporária como Stream
df_stream = spark.readStream.table("books_streaming_tmp_vw")

# 2. Exibe os dados em tempo real passando o checkpoint no Volume do UC
display(
    df_stream,
    checkpointLocation="/Volumes/demo_workspace/book_store/books/checkpoints/books_read_view"
)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW author_counts_tmp_vw AS (
  SELECT author, COUNT(book_id) AS book_count
  FROM books_streaming_tmp_vw
  GROUP BY author
)

In [0]:
%python
# 1. Cria a Temp View como um plano lógico de STREAMING
(spark.readStream
    .table("demo_workspace.book_store.books")
    .groupBy("author")
    .count()
    .createOrReplaceTempView("author_counts_tmp_vw")
)

# 2. Agora sim, lê a view em streaming e grava com writeStream
(spark.readStream.table("author_counts_tmp_vw")
    .writeStream
    .trigger(processingTime="10 seconds")
    .outputMode("complete")
    .option("checkpointLocation", "/Volumes/demo_workspace/book_store/books/checkpoints/author_counts")
    .table("author_counts")
)

In [0]:
%sql
insert into books
values
("B16","hands on deep learning","erica roque", "Computer Science", 25),
("B17","hands on deep ","erica roque", "Computer Science", 12)

In [0]:
%python
(spark.readStream.table("author_counts_tmp_vw")
    .writeStream
    .trigger(availableNow=True)
    .outputMode("complete")
    .option("checkpointLocation", "/Volumes/demo_workspace/book_store/books/checkpoints/author_counts")
    .table("author_counts")
    .awaitTermination()
)

In [0]:
%sql
select * from author_counts

In [0]:

df = spark.read.table("author_counts_tmp_vw") 
print(df.isStreaming)

In [0]:
%sql
select * from author_counts

In [0]:
inset